In [30]:
import math
import pandas as pd
import string
from collections import Counter

In [13]:
dataset = pd.read_csv('../data/sup100tweet.csv')

# Retrieve first 200 tweets as example
tweets = dataset['tweet']

# Merge all tweets by the same user
user_texts = dataset.groupby('username')['tweet'].agg(lambda x: ' '.join(x)).reset_index()

# Optional: rename columns for clarity
user_texts.columns = ['username', 'merged_tweet']
tweets = user_texts['merged_tweet']


# Simple preprocessing: lowercase, split by space, discarding mentions and URLs as they are too specific
docs = []
clean_tweets = []  
for tweet in tweets:
    if not tweet or tweet.strip() == "":
        continue
    words = tweet.lower().split()
    clean_words = [w for w in words if not w.startswith('@') and not w.startswith('http') and not w.startswith('www') ]
    if clean_words:
        docs.append(clean_words)
        clean_tweets.append(tweet)  # keep the original tweet for reference



# Calculate TF (normalized by max frequency in the document)
tf_list = []
for doc in docs:
    counts = Counter(doc)
    max_count = max(counts.values())
    tf = {word: count/max_count for word, count in counts.items()}
    tf_list.append(tf)

# Calculate IDF (Inverse Document Frequency)
all_words = set(word for doc in docs for word in doc)

N = len(docs)
idf = {}
for word in all_words:
    df = sum(1 for doc in docs if word in doc)
    idf[word] = math.log2(N/df) 

# Calculate TF-IDF
tfidf_list = []
for tf in tf_list:
    tfidf = {word: tf[word]*idf[word] for word in tf}
    tfidf_list.append(tfidf)


top_keywords_per_tweet = []

# Extract top keywords per tweet
for i, tfidf in enumerate(tfidf_list):
    top_keywords = sorted(tfidf.items(), key=lambda x: x[1], reverse=True)[:5]
    # Store as a dict with tweet and its top keywords
    top_keywords_per_tweet.append({
        'tweet': tweets[i],
        'username': user_texts['username'].iloc[i],
        'top_keywords': [(word, score) for word, score in top_keywords],
        'best_TF_IDF': top_keywords[0][1]
    })

# Sort tweets by the best TF-IDF score, descending
top_tweets = sorted(top_keywords_per_tweet, key=lambda x: x['best_TF_IDF'], reverse=True)[:5]

# Print them
for t in top_tweets:
    print(f"username: {t['username']}")
    print(f"Top keywords: {t['top_keywords']} \n")


FileNotFoundError: [Errno 2] No such file or directory: '../data/sup100tweet.csv'

In [ ]:
dataset = pd.read_csv('../data/horoscope_saved.csv')


# # Merge all tweets by the same user
# user_texts = dataset.groupby('username')['tweet'].agg(lambda x: ' '.join(x)).reset_index()

# # Optional: rename columns for clarity
# user_texts.columns = ['username', 'merged_tweet']
# tweets = user_texts['merged_tweet']




# # Simple preprocessing: lowercase, split by space, discarding mentions and URLs as they are too specific
# docs = []
# clean_tweets = []  
# for tweet in tweets:
#     if not tweet or tweet.strip() == "":
#         continue
#     words = tweet.lower().split()
#     clean_words = [w for w in words if not w.startswith('@') and not w.startswith('http') and not w.startswith('www') ]
#     if clean_words:
#         docs.append(clean_words)
#         clean_tweets.append(tweet)  # keep the original tweet for reference


# Getting the list of words with the lowest TF.IDF to filter it out

translator = str.maketrans('', '', string.punctuation)

predictions = dataset['horoscope']
docs_cleaned = predictions.apply(clean_text)
docs = [pred.lower().split() for pred in docs_cleaned ]


# # Calculate TF (normalized by max frequency in the document)
# tf_list = []
# for doc in docs:
#     counts = Counter(doc)
#     max_count = max(counts.values())
#     tf = {word: count/max_count for word, count in counts.items()}
#     tf_list.append(tf)

# # Calculate IDF (Inverse Document Frequency)
# all_words = set(word for doc in docs for word in doc)

# N = len(docs)
# idf = {}
# for word in all_words:
#     df = sum(1 for doc in docs if word in doc)
#     idf[word] = math.log2(N/df) 

# # Calculate TF-IDF
# tfidf_list = []
# for tf in tf_list:
#     tfidf = {word: tf[word]*idf[word] for word in tf}
#     tfidf_list.append(tfidf)


# top_keywords = sorted(tfidf.items(), key=lambda x: x[1], reverse=False)

# print(top_keywords[:50])




# enlever mots présents une seule fois


Weakest IDF :
[('weaker', 12.100596641013844), ('stabilizing', 12.100596641013844), ('tax', 12.100596641013844), ('defined', 12.100596641013844), ('comparing', 12.100596641013844), ('latent', 12.100596641013844), ('withhold', 12.100596641013844), ('refurnishing', 12.100596641013844), ('unfounded', 12.100596641013844), ('subside', 12.100596641013844), ('arguing', 12.100596641013844), ('oddball', 12.100596641013844), ('preach', 12.100596641013844), ('suspicion', 12.100596641013844), ('drivel', 12.100596641013844), ('someday', 12.100596641013844), ('settlement', 12.100596641013844), ('chose', 12.100596641013844), ('rhythmic', 12.100596641013844), ('haphazardly', 12.100596641013844), ('premonitions', 12.100596641013844), ('longawaited', 12.100596641013844), ('blunt', 12.100596641013844), ('presently', 12.100596641013844), ('graphics', 12.100596641013844), ('juicy', 12.100596641013844), ('deadline', 12.100596641013844), ('loom', 12.100596641013844), ('untapped', 12.100596641013844), ('narro

## Code for functions

In [38]:
def clean_text(text):

    text = text.lower()

    translator = str.maketrans('', '', string.punctuation)

    text_without_punct = text.translate(translator)
    
    return text_without_punct

def getting_idf(input_files):
    N = len(input_files)
    doc_freq = Counter()
    
    #IDF, we only want the number of docs the word is in
    for doc in input_files:
        unique_words_in_doc = set(doc) 
        doc_freq.update(unique_words_in_doc)

    # Computing IDF for every word
    idf = {}
    for word, freq in doc_freq.items():
        idf[word] = math.log2(N / freq)

    return idf
    

def getting_tf(input_files):
    tf_list = []
    for doc in input_files:
        counts = Counter(doc)
        max_count = max(counts.values())
        tf = {word: count/max_count for word, count in counts.items()}
        tf_list.append(tf)
    return(tf_list)

def getting_tf_idf(input_files):
    tf_list = getting_tf(input_files)
    idf_list = getting_idf(input_files)

    tfidf_list = []
    for tf in tf_list:
        tfidf = {word: tf[word]*idf_list[word] for word in tf}
        tfidf_list.append(tfidf)
    
    return(tfidf_list)
        
    
def stop_words(input_files, threshold): 

    idf = getting_idf(input_files)

    # Stop words identification
    sorted_idf = sorted(idf.items(), key=lambda x: x[1])

    idf_cut_off = math.log2(1 / threshold)

    cropped = [(word, score) for (word, score) in sorted_idf if score < idf_cut_off]

    stop_words = [word for word, score in cropped]

    # Les 50 mots les plus "inutiles" (présents partout)
    print(f"Most common words (present in more than {100*threshold} % of the dataset):")
    print(stop_words)
    print(len(stop_words))

    return(cropped, stop_words)


## Creating a whitelist for most specific words by categories

We suppress the birthday category from our analysis as it is always the same prediction for the same sign.

In [ ]:
dataset = pd.read_csv('../data/horoscope_full.csv')
dataset_filtered = dataset[dataset['category'] != 'birthday'].copy()

categories = dataset_filtered.groupby('category')['horoscope'].agg(lambda x: ' '.join(x)).reset_index()
docs_cleaned = categories['horoscope'].apply(clean_text)
cat_docs = [pred.split() for pred in docs_cleaned ]

cat_tf_idf = getting_tf_idf(cat_docs)

top_keywords_per_cat = []

blacklist = {'she', 'he', 'her','fourweek','workrelated', 'workweek', 'youd', 'todays', 'arrive', 'interesting', 'theyre', 'virgo', 'scorpio', 'cancer', 'libra', 'leo', 'gemini', 'taurus', 'aries', 'pisces', 'capricorn', 'aquarius', 'sagittarius'} 
# supressing the words that are the results of compression, or that are indeed to, common for being whitelisted
# also, we supress the sign which are the most repeated words of the general category. 
# We will anyway take these words with us when creating a whitelist by sign.
target_count = 15

whitelist_cat = []

# Extract top keywords per tweet
for i, tfidf in enumerate(cat_tf_idf):
    sorted_scores = sorted(tfidf.items(), key=lambda x: x[1], reverse=True)
    final_keywords = []

    for word, score in sorted_scores:

        if word in blacklist:
            continue 
        
        final_keywords.append((word, score))
        
        if len(final_keywords) == target_count:
            break
    
    # Store as a dict with tweet and its top keywords
    top_keywords_per_cat.append({
        'category': categories['category'].iloc[i],
        'top_keywords': [word for word, score in final_keywords],
        'best_TF_IDF': final_keywords[0][1]
    })

#print(top_keywords_per_cat)

whitelist_cat = set(word for category in top_keywords_per_cat for word in category['top_keywords'])

# [{'category': 'career', 'top_keywords': ['workplace', 'relations', 'tremendous', 'major', 'employers', 'coworkers', 'opposition', 'superiors', 'trend', 'jobs', 'concept', 'towel', 'four', 'andor', 'employer'], 'best_TF_IDF': 0.006939488672286022}, 
# {'category': 'general', 'top_keywords': ['virgo', 'scorpio', 'cancer', 'libra', 'leo', 'gemini', 'taurus', 'aries', 'pisces', 'capricorn', 'aquarius', 'sagittarius', 'income', 'visitors', 'youd'], 'best_TF_IDF': 0.049633389734912575}, 
# {'category': 'love', 'top_keywords': ['prospective', 'astral', 'configuration', 'celestial', 'date', 'indicates', 'concerning', 'indicate', 'concerned', 'constellation', 'permanent', 'fascinating', 'yourselves', 'sorted', 'interplay'], 'best_TF_IDF': 0.020040662213186174}, 
# {'category': 'wellness', 'top_keywords': ['transit', 'diet', 'yoga', 'drinking', 'aerobic', 'recommended', 'exercises', 'vegetables', 'bodys', 'digestive', 'wellness', 'lavender', 'liver', 'organs', 'dairy'], 'best_TF_IDF': 0.023549246750421495}]

{'opposition', 'four', 'jobs', 'concept', 'workplace', 'abrasive', 'major', 'date', 'concerned', 'organs', 'aerobic', 'income', 'celestial', 'constellation', 'indicate', 'legal', 'vegetables', 'recommended', 'astral', 'exercises', 'indicates', 'paperwork', 'yoga', 'superiors', 'drinking', 'permanent', 'repairs', 'fascinating', 'employers', 'liver', 'occult', 'bodys', 'affiliated', 'relatives', 'diet', 'interplay', 'relations', 'employer', 'coworkers', 'towel', 'concerning', 'studies', 'tremendous', 'configuration', 'intellectual', 'sorted', 'digestive', 'yourselves', 'andor', 'prospective', 'transit', 'wellness', 'dairy', 'trend', 'delayed', 'downside', 'lavender', 'visitors', 'members'}


## Creating a whitelist for most specific words by sign

In [ ]:
categories = dataset_filtered.groupby('sign')['horoscope'].agg(lambda x: ' '.join(x)).reset_index()
docs_cleaned = categories['horoscope'].apply(clean_text)
cat_docs = [pred.split() for pred in docs_cleaned ]

cat_tf_idf = getting_tf_idf(cat_docs)

top_keywords_per_sign = []

blacklist = {'prefers', 'isn’t'}
target_count = 11 # the sign name + 10 most specific words

whitelist_sign = []

# Extract top keywords per tweet
for i, tfidf in enumerate(cat_tf_idf):
    sorted_scores = sorted(tfidf.items(), key=lambda x: x[1], reverse=True)
    final_keywords = []

    for word, score in sorted_scores:

        if word in blacklist:
            continue 
        
        final_keywords.append((word, score))
        
        if len(final_keywords) == target_count:
            break
    
    # Store as a dict with tweet and its top keywords
    top_keywords_per_sign.append({
        'sign': categories['sign'].iloc[i],
        'top_keywords': [word for word, score in final_keywords],
        'best_TF_IDF': final_keywords[0][1]
    })

# print(top_keywords_per_sign)


whitelist_sign = set(word for sign in top_keywords_per_sign for word in sign['top_keywords'])

final_whitelist = whitelist_cat | whitelist_sign

print(len(whitelist_cat))
print(len(whitelist_sign))
print(len(final_whitelist))

#Total length of white_list: 189 words

# [{'sign': 'aquarius', 'top_keywords': ['aquarius', 'liberate', 'twisted', 'barnacles', 'prefers', 'infinite', 'italian', 'obey', 'plummet', 'crawl', 'puzzling', 'bangs'], 'best_TF_IDF': 0.23276278265551856}, 
# {'sign': 'aries', 'top_keywords': ['aries', 'refurnishing', 'midday', 'creeping', 'reborn', 'redecorating', 'remodeling', 'roiling', 'millionaire', 'sculpting', 'biting'], 'best_TF_IDF': 0.2352046231361471}, 
# {'sign': 'cancer', 'top_keywords': ['cancer', 'mileage', 'responding', 'deliveries', 'freedomloving', 'unreal', 'protest', 'portfolio', 'impatiently', 'vindictive', 'nonexistent'], 'best_TF_IDF': 0.2433454592625667}, 
# {'sign': 'capricorn', 'top_keywords': ['capricorn', 'aloud', 'escapism', 'ray', 'faraway', 'requested', 'salesperson', 'dab', 'restraints', 'constitutes', 'inferno'], 'best_TF_IDF': 0.23550338455029815}, 
# {'sign': 'gemini', 'top_keywords': ['gemini', 'random', 'gray', 'factoriented', 'psychologist', 'rulers', 'strategically', 'rebalancing', 'inherited', 'conventions', 'disappeared'], 'best_TF_IDF': 0.24465621523027387}, 
# {'sign': 'leo', 'top_keywords': ['leo', 'drawers', 'donating', 'earning', 'pigeonholed', 'attribute', 'bucks', 'studious', 'transaction', 'myth', 'tranquil'], 'best_TF_IDF': 0.24266570252815378}, 
# {'sign': 'libra', 'top_keywords': ['libra', 'makeup', 'hardships', 'lily', 'rearing', 'entice', 'distasteful', 'pupils', 'renovation', 'prisoner', 'throwback'], 'best_TF_IDF': 0.23908963478209716}, 
# {'sign': 'pisces', 'top_keywords': ['pisces', 'characters', 'exhibitions', 'stampede', 'frolic', 'palaces', 'timeless', 'messenger', 'storms', 'liberties', 'postcard'], 'best_TF_IDF': 0.2348289390007269}, 
# {'sign': 'sagittarius', 'top_keywords': ['sagittarius', 'outs', 'fidget', 'halftruths', 'braver', 'unplug', 'beckon', 'honeys', 'fallow', 'activating', 'pampered'], 'best_TF_IDF': 0.10607651468241104}, 
# {'sign': 'scorpio', 'top_keywords': ['scorpio', 'isn’t', 'battlefield', 'answering', 'diplomat', 'niggling', 'regretting', 'reassess', 'ramming', 'mischief', 'organizational', 'undermining'], 'best_TF_IDF': 0.24573773506705926}, 
# {'sign': 'taurus', 'top_keywords': ['taurus', 'gateway', 'placid', 'midday', 'declaring', 'reborn', 'rocking', 'strayed', 'widely', 'affirm', 'bar'], 'best_TF_IDF': 0.23465209095629386}, 
# {'sign': 'virgo', 'top_keywords': ['virgo', 'engrossed', 'substantiate', 'comedy', 'scientists', 'escalates', 'stupid', 'indicative', 'giddy', 'substantial', 'hamburger'], 'best_TF_IDF': 0.24588986754751502}]

59
130
189


## Getting the stop words


In [46]:
predictions = dataset_filtered['horoscope']
docs_cleaned = predictions.apply(clean_text)
docs = [pred.lower().split() for pred in docs_cleaned ]

(idf_pairs, common_words) = stop_words(docs, 0.10)

rescued_keywords = final_whitelist & set(common_words)
final_stop_words = set(common_words) - final_whitelist

print(len(rescued_keywords), " words rescued from being banned: ", rescued_keywords)
print(len(final_stop_words), " words that will be suppressed from predictions: ", final_stop_words)

Most common words (present in more than 10.0 % of the dataset):
['you', 'to', 'the', 'your', 'and', 'a', 'of', 'is', 'be', 'in', 'that', 'this', 'it', 'will', 'are', 'with', 'for', 'have', 'may', 'today', 'on', 'if', 'or', 'not', 'yourself', 'time', 'can', 'but', 'dont', 'as', 'do', 'at', 'get', 'feel', 'more', 'what', 'out', 'take', 'make', 'some', 'could', 'about', 'so', 'need', 'up', 'from', 'day', 'all', 'an', 'there', 'energy', 'way', 'one', 'find', 'people', 'good', 'by', 'work', 'into', 'things', 'others', 'than', 'when', 'someone', 'just', 'its', 'go', 'life', 'want', 'something', 'try', 'now', 'them', 'other', 'like', 'new', 'might', 'planetary', 'very', 'todays', 'much', 'love', 'they', 'how', 'been']
85
0  words rescued from being banned:  set()
85  words that will be suppressed from predictions:  {'energy', 'something', 'that', 'into', 'way', 'and', 'out', 'what', 'today', 'they', 'may', 'up', 'them', 'if', 'good', 'life', 'make', 'for', 'of', 'get', 'work', 'to', 'dont', '

## Rewriting all predictions in the dataset, without the stop words